# Notebook 6.2: Implementing and Evaluating Simple Linear Regression with Scikit-learn

**Companion to Chapter 6: Simple Linear Regression**  
*Machine Learning with Python: Principles and Practical Techniques*

> **Estimated time:** 40–50 minutes  
> **Level:** Beginner  
> **Environment:** Google Colab or Jupyter Notebook

---

## Related chapter ideas

This notebook applies simple linear regression using Scikit-learn. It connects the hypothesis, slope, intercept, residuals, and cost concepts from Notebook 6.1 to a complete model-development workflow.

## Learning objectives

By the end of this notebook, you will be able to:

1. prepare one feature and one continuous target for Scikit-learn;
2. create reproducible training and test sets;
3. fit a `LinearRegression` model;
4. interpret the learned intercept and coefficient;
5. generate and visualize predictions;
6. evaluate predictions using MAE, MSE, RMSE, and $R^2$;
7. inspect residuals and important linear-regression assumptions; and
8. communicate predictions and limitations responsibly.


## What will you build?

You will train a model that uses research experience to predict annual stipend. The workflow is:

**Inspect → Select $X$ and $y$ → Split → Fit → Predict → Evaluate → Diagnose → Interpret**

> **Responsible practice:** This synthetic example demonstrates a statistical method. Experience alone is insufficient for real compensation decisions, which require attention to role, discipline, responsibilities, location, funding, equity, and applicable policy.


## 1. Import the libraries


In [ ]:
from io import StringIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

pd.set_option("display.precision", 3)
print("Pandas version:", pd.__version__)


## 2. Load and inspect the dataset


In [ ]:
stipend_csv = """experience_years,annual_stipend_thousands
0.5,34
1.0,37
1.5,39
2.0,43
2.5,45
3.0,49
3.5,50
4.0,54
4.5,57
5.0,60
5.5,61
6.0,66
6.5,68
7.0,70
7.5,74
8.0,77
8.5,79
9.0,82
9.5,85
10.0,88
"""

stipends = pd.read_csv(StringIO(stipend_csv))
print("Dataset shape:", stipends.shape)
display(stipends.head())


In [ ]:
print("Dataset shape:", stipends.shape)
display(stipends.head())
stipends.info()


In [ ]:
quality_check = pd.DataFrame({
    "dtype": stipends.dtypes.astype(str),
    "missing": stipends.isna().sum(),
    "unique": stipends.nunique(),
})

display(quality_check)
print("Duplicate rows:", stipends.duplicated().sum())


Always inspect data before modeling. This dataset contains 20 complete, unique observations and two numerical columns.


## 3. Visualize the relationship


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(
    stipends["experience_years"],
    stipends["annual_stipend_thousands"],
    color="#4C78A8", edgecolor="black", s=75, alpha=0.85,
)
ax.set_title("Research Experience and Annual Stipend")
ax.set_xlabel("Research experience (years)")
ax.set_ylabel("Annual stipend ($ thousands)")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


The scatter plot suggests a strong positive, approximately linear relationship. This supports trying simple linear regression, but a visual pattern alone does not validate every model assumption.


## 4. Prepare the feature matrix and target


In [ ]:
# X must be two-dimensional for Scikit-learn: (rows, features).
X = stipends[["experience_years"]]

# y is a one-dimensional target vector.
y = stipends["annual_stipend_thousands"]

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())
display(y.head().to_frame())


Double brackets preserve `X` as a two-dimensional DataFrame. Scikit-learn expects feature matrices in the shape `(number of observations, number of features)` even when there is only one feature.


## 5. Create training and test sets


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
)

print("Training observations:", len(X_train))
print("Test observations:", len(X_test))

split_preview = pd.DataFrame({
    "experience_years": X_test["experience_years"],
    "actual_stipend": y_test,
}).sort_values("experience_years")
display(split_preview)


- The model learns from the **training set**.
- The **test set** estimates performance on unseen examples.
- `random_state=42` makes the split reproducible.

With only 20 observations, the test set contains five cases. Its metrics are useful for teaching but too unstable for high-stakes conclusions.


## 6. Create and train the model


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully.")


`fit()` learns the intercept and coefficient from the training data by minimizing the sum of squared residuals. Scikit-learn's `LinearRegression` uses an ordinary least-squares solver; we do not need to write the optimization procedure ourselves to apply the model.


## 7. Inspect and interpret the learned parameters


In [ ]:
intercept = model.intercept_
slope = model.coef_[0]

print("Intercept:", round(intercept, 3))
print("Slope:", round(slope, 3))
print(f"Fitted equation: stipend = {intercept:.3f} + {slope:.3f} × experience")


Interpretation:

- **Slope:** Each additional year of research experience is associated with an average predicted stipend increase of approximately the slope multiplied by $1,000.
- **Intercept:** The model's predicted stipend at zero years of experience. It should be interpreted cautiously because zero lies just outside the observed range, which starts at 0.5 years.

The slope describes association in this dataset; it does not prove a causal effect.


In [ ]:
print(f"Predicted average increase per additional year: ${slope * 1000:,.0f}")
print(f"Predicted stipend at zero years: ${intercept * 1000:,.0f}")


## 8. Generate predictions


In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

test_results = pd.DataFrame({
    "experience_years": X_test["experience_years"],
    "actual_stipend": y_test,
    "predicted_stipend": y_test_pred,
})
test_results["residual"] = (
    test_results["actual_stipend"] - test_results["predicted_stipend"]
)

display(test_results.sort_values("experience_years"))


## 9. Plot the fitted regression line


In [ ]:
x_line = pd.DataFrame({
    "experience_years": np.linspace(X["experience_years"].min(), X["experience_years"].max(), 200)
})
y_line = model.predict(x_line)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_train["experience_years"], y_train,
           color="#4C78A8", edgecolor="black", s=70, label="Training data")
ax.scatter(X_test["experience_years"], y_test,
           color="#F58518", edgecolor="black", marker="s", s=80, label="Test data")
ax.plot(x_line["experience_years"], y_line,
        color="#E45756", linewidth=2.5, label="Fitted regression line")
ax.set_title("Simple Linear Regression Model")
ax.set_xlabel("Research experience (years)")
ax.set_ylabel("Annual stipend ($ thousands)")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


The line was learned from training data. Test points are shown for evaluation, not for adjusting the line.


## 10. Evaluate prediction errors


We use complementary metrics:

- **MAE:** average absolute error, in the original target units;
- **MSE:** average squared error, which penalizes large errors more strongly;
- **RMSE:** square root of MSE, returned to the original target units; and
- **$R^2$:** proportion of target variance explained relative to predicting the test-set mean.

Lower MAE, MSE, and RMSE are better. An $R^2$ closer to 1 indicates stronger fit, but it does not prove causation or real-world validity.


In [ ]:
mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_test_pred)

metrics = pd.DataFrame({
    "metric": ["MAE", "MSE", "RMSE", "R²"],
    "value": [mae, mse, rmse, r2],
    "interpretation_unit": [
        "$ thousands", "squared $ thousands", "$ thousands", "unitless"
    ],
})

display(metrics)
print(f"Typical test error (RMSE): approximately ${rmse * 1000:,.0f}")


### Do not compare these values as if they share a scale

MSE is squared, MAE and RMSE use the target's units, and $R^2$ is unitless. Each answers a different question.


## 11. Compare training and test performance


In [ ]:
performance_comparison = pd.DataFrame({
    "dataset": ["Training", "Test"],
    "MAE": [
        mean_absolute_error(y_train, y_train_pred),
        mean_absolute_error(y_test, y_test_pred),
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_train, y_train_pred)),
        rmse,
    ],
    "R²": [
        r2_score(y_train, y_train_pred),
        r2,
    ],
})

display(performance_comparison)


A large gap between training and test performance can indicate poor generalization. With a very small test set, however, differences may also reflect sampling variation.


## 12. Examine residuals


In [ ]:
test_residuals = y_test.to_numpy() - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].scatter(y_test_pred, test_residuals,
                color="#4C78A8", edgecolor="black", s=75)
axes[0].axhline(0, color="#E45756", linestyle="--", linewidth=2)
axes[0].set_title("Test Residuals vs. Predictions")
axes[0].set_xlabel("Predicted stipend ($ thousands)")
axes[0].set_ylabel("Residual: actual − predicted")
axes[0].grid(alpha=0.2)

axes[1].hist(test_residuals, bins=5, color="#54A24B", edgecolor="black")
axes[1].axvline(0, color="#E45756", linestyle="--", linewidth=2)
axes[1].set_title("Distribution of Test Residuals")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()


Residuals should ideally form an unstructured cloud around zero. We look for:

- curvature, which may indicate nonlinearity;
- increasing or decreasing spread, which may indicate non-constant variance;
- extreme residuals, which may indicate unusual or influential observations; and
- dependence across observations, which requires knowledge of how data were collected.

Five test residuals are far too few for a reliable distributional diagnosis. This plot demonstrates the method, not a definitive assumption check.


## 13. Understand the main assumptions


| Assumption | Meaning | How to investigate |
|---|---|---|
| Linearity | Mean outcome changes approximately linearly with $x$ | Scatter and residual plots |
| Independent observations | One observation does not determine another | Study design and collection process |
| Constant error variance | Residual spread is reasonably stable | Residuals versus fitted values |
| Approximately normal errors | Important mainly for classical intervals and tests | Histogram or Q–Q plot with adequate data |
| Limited influential points | Results are not controlled by a few cases | Residual, leverage, and influence analysis |

Good predictive performance does not automatically prove that all assumptions hold.


## 14. Predict a new value


In [ ]:
new_researcher = pd.DataFrame({"experience_years": [6.0]})
new_prediction = model.predict(new_researcher)[0]

print(f"Predicted annual stipend for 6 years of experience: ${new_prediction * 1000:,.0f}")


The prediction is a model estimate, not a guaranteed or recommended stipend. A responsible report should state the input, units, training-data range, uncertainty, and important omitted factors.


## 15. Examine interpolation and extrapolation


In [ ]:
prediction_cases = pd.DataFrame({
    "experience_years": [4.0, 12.0, 25.0],
    "type": ["Interpolation", "Extrapolation", "Extreme extrapolation"],
})
prediction_cases["predicted_stipend_thousands"] = model.predict(
    prediction_cases[["experience_years"]]
)

display(prediction_cases)


The model always returns a numerical result, but that does not make every result trustworthy. Predictions beyond the observed 0.5–10 year range assume that the same straight-line relationship continues.


## 16. Explore sensitivity to the random split


In [ ]:
split_results = []

for seed in range(10):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=seed
    )
    candidate_model = LinearRegression().fit(X_tr, y_tr)
    candidate_predictions = candidate_model.predict(X_te)
    split_results.append({
        "random_state": seed,
        "slope": candidate_model.coef_[0],
        "intercept": candidate_model.intercept_,
        "test_RMSE": np.sqrt(mean_squared_error(y_te, candidate_predictions)),
        "test_R²": r2_score(y_te, candidate_predictions),
    })

split_table = pd.DataFrame(split_results)
display(split_table)


Metrics vary because each small test set contains different observations. In larger workflows, cross-validation provides a more stable evaluation than relying on one split. It will be introduced with broader model-selection methods.


## 17. Guided practice


Complete these tasks:

1. Use `model.coef_` to report the predicted stipend change for two additional years of experience.
2. Predict the stipend for 7.5 years of experience.
3. Identify the test observation with the largest absolute residual.
4. Calculate the test MAE manually using NumPy.
5. Explain why the model must not be refitted using the test set before reporting test performance.


In [ ]:
# Write your solution here.


<details>
<summary><strong>Open the suggested solution</strong></summary>

```python
# 1. Predicted change for two years
print(2 * model.coef_[0] * 1000)

# 2. Prediction for 7.5 years
example = pd.DataFrame({"experience_years": [7.5]})
print(model.predict(example)[0])

# 3. Largest absolute residual
largest_index = test_results["residual"].abs().idxmax()
display(test_results.loc[[largest_index]])

# 4. Manual MAE
manual_mae = np.mean(np.abs(y_test.to_numpy() - y_test_pred))
print(manual_mae)

# 5. Refitting on test data would contaminate the independent evaluation and
# produce an optimistically biased performance estimate.
```

</details>


## 18. Challenge: Communicate the model responsibly


Write a short model report containing:

1. the prediction question;
2. the fitted equation with units;
3. the training and test sample sizes;
4. MAE, RMSE, and $R^2$;
5. one residual-based observation;
6. the observed experience range;
7. one limitation of the dataset; and
8. one statement preventing causal or high-stakes misuse.

Avoid claiming that experience alone determines a fair stipend.


## 19. Common mistakes to avoid


| Mistake | Why it is a problem | Better practice |
|---|---|---|
| Passing a one-dimensional `X` | Scikit-learn expects a feature matrix | Use `data[["feature"]]` |
| Training on all data before testing | Leaves no independent evaluation | Split before calling `fit()` |
| Adjusting the model after viewing test results repeatedly | Test set becomes part of development | Preserve a final independent evaluation |
| Reporting only $R^2$ | It hides error magnitude in target units | Report multiple complementary metrics |
| Treating association as causation | The design does not establish cause | Use associational language |
| Trusting extreme extrapolations | Relationship may change outside the data range | State and respect the observed range |


## 20. Reflection


1. Why must `X` be two-dimensional in Scikit-learn?
2. What do `intercept_` and `coef_` represent?
3. How do MAE, MSE, RMSE, and $R^2$ differ?
4. What pattern would you hope to see in a residual plot?
5. Why are the test metrics unstable in this notebook?
6. Why is a high $R^2$ insufficient to justify a real compensation policy?


## 21. Key takeaways


- Scikit-learn implements ordinary least-squares regression through `LinearRegression`.
- A proper workflow separates training data used for learning from test data used for evaluation.
- `intercept_` and `coef_` define the fitted line and must be interpreted with units and context.
- MAE and RMSE express error in target units, MSE emphasizes larger errors, and $R^2$ describes relative variance explained.
- Residual plots help reveal nonlinearity, unequal variance, and unusual observations.
- Small test sets produce unstable metrics; one split should not support strong conclusions.
- Model predictions are associations and estimates—not causal findings, guarantees, or compensation policies.

### Looking ahead

In **Notebook 6.3: Exploring Gradient Descent Visually**, you will connect Scikit-learn's fitted parameters to the chapter's optimization theory by watching the parameters move down the cost surface under different learning rates.
